# Fine-tuning with context: a conversation, not an exchange

Notebook 6_1 trained on one `You` turn and the `Assistant` reply to it, padded out to
`block_size`. That teaches the model to answer a message. It cannot teach it to *follow a
conversation*, because every training sample threw the history away.

This notebook changes one thing: instead of one exchange per row, consecutive exchanges are
**packed** into a single `block_size` window until it is full.

```
6_1   [ You A | Assistant A | <pad> <pad> <pad> ... <pad> ]     one exchange, 75% padding
6_2   [ You A | Assistant A | You B | Assistant B | You C | ... ]   as many as fit
```

That single change does three things, and only the third is the point:

| | 6_1 | 6_2 |
| --- | --- | --- |
| rows | 4,061 | **1,191** |
| positions carrying loss | 24.8% | **84.7%** |
| steps per epoch | 121 | **35** |

The same corpus in **3.4x fewer steps**, because three quarters of every 6_1 batch was
padding being computed and then thrown away by the mask. That is a real efficiency win and
it is worth understanding, but it is a side effect.

**The actual point is that the model can now see what was said earlier.** Within a window,
predicting the reply to `You C` has `You A / Assistant A / You B / Assistant B` in front of
it. Whether a 13.8M model trained on 176k tokens *uses* that history is a measurable
question rather than an assumption, and there is a cell near the end that measures it two
ways — one that appears to show a large benefit and one that controls for the obvious
confound and shows a small one. The small number is the true one.

| Stage | What it produces |
| --- | --- |
| Load | the notebook 5 dataset, and the tokenizer that wrote it |
| Check | that the corpus alternates, before anything depends on it |
| Pair | each `You` turn joined to its `Assistant` reply |
| Pack | consecutive exchanges concatenated up to `block_size` |
| Mask | targets where padding is ignored by the loss |
| Split | train / validation on whole windows |
| Restore | notebook 4's checkpoint, with its own saved hyperparameters |
| Fine-tune | with evaluation, early stopping, and a best-checkpoint save |
| Talk | a multi-turn conversation that carries its own history |

## Imports and paths

Both the tokenizer (`minbpe`) and the model (`transformer/model.py`) live outside this
folder, so the path setup has to happen before anything else. `sys.path.append('..')` alone
reaches `transformer` but not `minbpe`, which is the failure this cell exists to avoid.

In [ ]:
import sys
from pathlib import Path

# The tutorial's own package (transformer/) sits one level up from Notebooks/.
tutorial_root = Path.cwd().parent
sys.path.insert(0, str(tutorial_root))

try:
    import minbpe
except ModuleNotFoundError:
    for repo_root in (Path.cwd(), *Path.cwd().parents):
        if (repo_root / "minbpe" / "minbpe" / "base.py").exists():
            sys.path.insert(0, str(repo_root / "minbpe"))
            print("using minbpe clone at:", repo_root / "minbpe")
            break
    else:
        raise ModuleNotFoundError(
            "minbpe not found. Install it with "
            "`pip install git+https://github.com/karpathy/minbpe.git`"
        )
else:
    print("using installed minbpe:", Path(minbpe.__file__).parent)

print("tutorial root:", tutorial_root)

In [ ]:
import json
import math
import time

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

from minbpe import RegexTokenizer
from transformer.model import GPTLanguageModel

torch.manual_seed(3647)

tokenizer = RegexTokenizer()
tokenizer.load(model_file="../output/tokenizer/my_tokenizer.model")


def get_vocab_size(tokenizer: RegexTokenizer) -> int:
    """Number of embedding rows the model needs: one past the largest usable id.

    Not len(vocab) + len(special_tokens). load() rebuilds .vocab with the special tokens
    already folded in, so that expression counts them twice - 1029 on a freshly trained
    tokenizer but 1034 on a reloaded one. The notebook 4 checkpoint was saved with 1029
    embedding rows, so the wrong number makes load_state_dict() fail on a shape mismatch.
    """
    largest = max(tokenizer.vocab)
    if tokenizer.special_tokens:
        largest = max(largest, *tokenizer.special_tokens.values())
    return largest + 1


vocab_size = get_vocab_size(tokenizer)
print(f"vocab_size = {vocab_size}")

## Configuration

`block_size`, `n_embd`, `n_head`, `n_layer` and `vocab_size` are not set here — they are read
out of the checkpoint below, because a fine-tuning run that disagrees with the model it is
loading does not train badly, it fails to load.

**`max_epochs` is much larger than 6_1's.** Packing cut an epoch from 121 steps to 35, so
three epochs would now be barely a hundred optimiser steps. Twelve epochs here is roughly the
same amount of *training* as three epochs there, on the same data — the unit changed, not the
work. Early stopping decides when to actually quit.

In [ ]:
# --- Paths ----------------------------------------------------------------
DATASET_PATH = Path("../output/fine_tuning/data/fine_tuning.json")
PRETRAINED_CHECKPOINT = Path("../output/pre_training/run_4/checkpoint.pth")
OUTPUT_CHECKPOINT = Path("../output/fine_tuning/with_context/checkpoint.pth")

# --- The sample format, as notebook 5 wrote it ----------------------------
START_OF_TEXT = "<|startoftext|>"
SEPARATOR = "<|separator|>"
END_OF_TEXT = "<|endoftext|>"
PADDING = "<|padding|>"
USER_ROLE = "You"
ASSISTANT_ROLE = "Assistant"

# torch's cross_entropy ignores this target value by default.
IGNORE_INDEX = -100

# --- Training -------------------------------------------------------------
batch_size = 32          # notebook 4 measured 64 as a memory cliff on MPS, not a 2x cost
learning_rate = 6e-5
grad_clip = 1.0
val_fraction = 0.05

eval_interval = 20
eval_batches = 15
max_epochs = 12          # 35 steps/epoch after packing; see the note above
max_steps = None         # None = run all epochs; set an integer for a quick smoke test
early_stop_patience = 5

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"       # without this branch an Apple machine silently trains on CPU
else:
    device = "cpu"

print(f"device = {device}")

In [ ]:
if not PRETRAINED_CHECKPOINT.exists():
    raise FileNotFoundError(
        f"{PRETRAINED_CHECKPOINT.resolve()} not found - run "
        "4_1_ModelTrainingAllBatches.ipynb first, or point PRETRAINED_CHECKPOINT elsewhere"
    )

checkpoint = torch.load(PRETRAINED_CHECKPOINT, map_location=device, weights_only=False)
config = checkpoint["config"]
block_size = config["block_size"]

assert config["vocab_size"] == vocab_size, (
    f"checkpoint was trained with vocab_size={config['vocab_size']} but this tokenizer "
    f"gives {vocab_size} - the embedding table would not fit"
)

print(f"{PRETRAINED_CHECKPOINT.name}: epoch {checkpoint['epoch']}, "
      f"step {checkpoint['step']}, pre-training loss {checkpoint['loss']:.4f}")
for key, value in config.items():
    print(f"  {key:<12} {value}")

## The data

A flat JSON list of strings, alternating `You`, `Assistant`, `You`, ... all the way through.
Notebook 5 guarantees that ordering, and this notebook depends on it twice over: once to pair
a prompt with its reply, and again to pack pairs in the order they were actually said.

In [ ]:
if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f"{DATASET_PATH.resolve()} not found - run 5_FineTuningDataset.ipynb first"
    )

samples = json.loads(DATASET_PATH.read_text(encoding="utf-8"))

assert samples, "the dataset is empty"
assert len(samples) % 2 == 0, f"odd sample count ({len(samples):,}) - they cannot all pair up"

roles = [s[len(START_OF_TEXT):].split(SEPARATOR, 1)[0] for s in samples]
expected = [USER_ROLE, ASSISTANT_ROLE] * (len(samples) // 2)
first_break = next(
    (i for i, (actual, wanted) in enumerate(zip(roles, expected)) if actual != wanted), None
)
assert first_break is None, (
    f"alternation breaks at sample {first_break}: expected {expected[first_break]!r}, "
    f"got {roles[first_break]!r} - re-run 5_FineTuningDataset.ipynb"
)

for marker in (START_OF_TEXT, SEPARATOR, END_OF_TEXT, PADDING):
    assert marker in tokenizer.special_tokens, (
        f"{marker} is not a special token - re-run 2_BytePairEncoding.ipynb"
    )

start_token = tokenizer.special_tokens[START_OF_TEXT]
separator_token = tokenizer.special_tokens[SEPARATOR]
padding_token = tokenizer.special_tokens[PADDING]
eos_token = tokenizer.special_tokens[END_OF_TEXT]

print(f"{len(samples):,} samples ({len(samples) // 2:,} exchanges), alternating cleanly")
print(f"  {PADDING:<16} id {padding_token}")
print(f"  {END_OF_TEXT:<16} id {eos_token}   <- eos_token_id at generation time")

## Pairing, then packing

Two steps, and it matters that they are separate.

**Pair** is 6_1's step with one deliberate difference: an exchange longer than `block_size`
is *dropped* rather than truncated. 6_1 clips it from the front, which is correct there —
each exchange owns a row, so clipping costs part of its own question and nothing else. Here
exchanges share a window, and a clipped one arrives with its opening `<|startoftext|>` shaved
off, putting a sentence fragment at a window boundary and teaching the model that a window
can begin mid-turn. **Two of 4,061 exchanges are affected**, so dropping them is free; the
strict check below would have caught it either way, and did.

**Pack** is the new part: walk the exchanges in order and keep concatenating them into the
current window until the next one would not fit, then start a new window. Nothing is split
across windows — an exchange is atomic, so the model never sees half an answer.

Greedy packing is used rather than anything cleverer because the order carries meaning here.
Sorting exchanges by length would pack more tightly and destroy the thing being taught: the
exchanges in a window have to be the ones that actually followed each other.

**One honest defect.** Notebook 5 writes a flat list with no record of which chat each
exchange came from, so packing cannot tell where one conversation ends and the next begins.
Measured on this dataset, **15 of 1,191 windows (1.26%) straddle two unrelated chats** — the
tail of `book_club` sitting in front of the opening of `bro_chat` as if it were context. At
1.26% this is noise rather than a problem, but it is a real limitation of the format, not
something the packing code could fix. The way to remove it is for notebook 5 to record chat
boundaries; that is listed at the end rather than done here.

In [ ]:
def build_pairs(samples: list[str], block_size: int) -> tuple[list[list[int]], int]:
    """Encode each You -> Assistant exchange into one sequence of token ids.

    Exchanges longer than block_size are *dropped*, not truncated. 6_1 truncates them from
    the front, which is right there: each exchange has a row to itself, so a clipped one
    only costs part of its own question. Here it would be pasted into a shared window with
    its opening <|startoftext|> shaved off, putting a sentence fragment at a window
    boundary and teaching the model that a window can begin mid-turn. Two exchanges out of
    4,061 are affected, so dropping them costs nothing worth keeping.
    """
    sequences, dropped = [], 0
    for index in range(0, len(samples), 2):
        prompt = tokenizer.encode(samples[index], allowed_special="all")
        reply = tokenizer.encode(samples[index + 1], allowed_special="all")
        sequence = prompt + reply

        if len(sequence) > block_size:
            dropped += 1
            continue

        sequences.append(sequence)
    return sequences, dropped


def pack(sequences: list[list[int]], block_size: int) -> list[list[int]]:
    """Concatenate consecutive exchanges into windows of at most block_size tokens.

    An exchange is never split across two windows: if it does not fit, the current window
    is closed and it starts the next one. That keeps every window a sequence of complete
    turns, which is what makes the history in front of a reply meaningful.
    """
    windows, current = [], []
    for sequence in sequences:
        if current and len(current) + len(sequence) > block_size:
            windows.append(current)
            current = []
        current = current + sequence if current else list(sequence)
    if current:
        windows.append(current)
    return windows


exchanges, dropped = build_pairs(samples, block_size)
windows = pack(exchanges, block_size)

exchange_tokens = sum(len(e) for e in exchanges)
print(f"{len(samples) // 2:,} exchanges, {dropped} dropped for exceeding block_size")
print(f"{len(exchanges):,} exchanges -> {len(windows):,} windows of <= {block_size} tokens")
print(f"  {len(exchanges) / len(windows):.1f} exchanges per window on average")
print(f"  {exchange_tokens / (len(windows) * block_size):.1%} of the padded tensor is real "
      f"text (6_1 packed one exchange per row and reached "
      f"{exchange_tokens / (len(exchanges) * block_size):.1%})")

### Check the packing

Packing is easy to get subtly wrong in ways that do not raise: an exchange split across two
windows, a window over `block_size`, or - the one that would quietly undo the whole notebook -
tokens reordered so the "history" is not what was actually said.

The last assertion is the one that matters: flattening the windows back out must reproduce
the exchange list exactly, in order. If that holds, packing moved boundaries and nothing else.

In [ ]:
assert all(len(w) <= block_size for w in windows), "a window exceeds block_size"
assert all(len(w) > 0 for w in windows), "an empty window was produced"

# Every window opens a turn, and closes one.
for i, w in enumerate(windows):
    assert w[0] == start_token, f"window {i} does not open with <|startoftext|>"
    assert w[-1] == eos_token, f"window {i} does not end with <|endoftext|>"

# Nothing was split, reordered, dropped or duplicated: flattening must round-trip.
flat_windows = [t for w in windows for t in w]
flat_exchanges = [t for e in exchanges for t in e]
assert flat_windows == flat_exchanges, (
    "packing changed the token stream - an exchange was split or reordered"
)

# Each window holds a whole number of turns: <|startoftext|> appears in You/Assistant pairs.
for i, w in enumerate(windows):
    opens = sum(1 for t in w if t == start_token)
    assert opens % 2 == 0, f"window {i} holds {opens} turns - not whole exchanges"
    assert sum(1 for t in w if t == separator_token) == opens, f"window {i} lost a separator"

counts = [sum(1 for t in w if t == start_token) // 2 for w in windows]
print(f"{len(windows):,} windows, all <= {block_size} tokens, all opening and closing a turn")
print(f"flattened round-trip: {len(flat_windows):,} tokens identical to the exchange stream")
print(f"exchanges per window: min {min(counts)}, median "
      f"{sorted(counts)[len(counts) // 2]}, max {max(counts)}")

## Padding, and the loss mask

Identical in principle to 6_1, and worth restating because packing changes how much it
matters. `F.cross_entropy` ignores target positions equal to `-100` — that is what
`ignore_index` defaults to — so the mask lives in the *targets*, not in the model:

```
x  <|startoftext|> You <|separator|> hey <|endoftext|> ... <|padding|>
y  You <|separator|> hey <|endoftext|> ...        -100        -100
```

The original version of this notebook described this masking in a comment and then did not
do it: it passed `ignore_index=` to `GPTLanguageModel`, which is not a constructor argument,
and built `y` by shifting and appending a real `<|padding|>` id — so padding was scored as an
ordinary token to predict.

Packing makes that bug cheaper and the fix less dramatic. In 6_1, 75% of every batch was
padding, so masking it was the difference between training and not. Here only about 15% is,
which is precisely why it is worth keeping the check: a bug that costs you 15% is one you can
live with for a long time without noticing.

**What is deliberately not masked**: the prompts. Every `You` turn is scored too, not just
the replies. Masking prompts is a real refinement and it is listed at the end.

In [ ]:
def pad_and_mask(
    sequences: list[list[int]], block_size: int, padding_token: int
) -> tuple[torch.Tensor, torch.Tensor]:
    """Build (inputs, targets) where padding is masked out of the loss."""
    inputs = torch.full((len(sequences), block_size), padding_token, dtype=torch.long)
    targets = torch.full((len(sequences), block_size), IGNORE_INDEX, dtype=torch.long)

    for row, ids in enumerate(sequences):
        length = len(ids)
        sequence = torch.tensor(ids, dtype=torch.long)
        inputs[row, :length] = sequence
        # Target t is the token after input t. The last real token has no successor, so
        # positions length-1 onwards keep IGNORE_INDEX.
        targets[row, :length - 1] = sequence[1:]

    return inputs, targets

### Check the mask

Three assertions; the third proves the claim rather than restating it. The loss is computed
twice on random logits — once the way the model computes it, once by hand over only the
unmasked entries. If `-100` were doing nothing, the two would differ.

In [ ]:
mask_check = [
    [start_token, 10, 11, separator_token, 12, eos_token],   # 6 real tokens
    [start_token, 20, separator_token, eos_token],           # 4 real tokens
]
x_check, y_check = pad_and_mask(mask_check, block_size=8, padding_token=padding_token)

# 1. targets are inputs shifted left by one, wherever they are not masked
for row, ids in enumerate(mask_check):
    for position in range(len(ids) - 1):
        assert y_check[row, position] == x_check[row, position + 1], (
            f"target misaligned at row {row}, position {position}"
        )

# 2. padding is masked, real tokens (bar the last) are not
for row, ids in enumerate(mask_check):
    assert (y_check[row, len(ids) - 1:] == IGNORE_INDEX).all(), (
        f"row {row} leaves padding unmasked - the model would learn to predict it"
    )
    assert (y_check[row, :len(ids) - 1] != IGNORE_INDEX).all(), f"row {row} masks a real token"

# 3. the mask actually removes those positions from the average
torch.manual_seed(0)
logits = torch.randn(len(mask_check), 8, vocab_size)
as_model_computes_it = F.cross_entropy(logits.view(-1, vocab_size), y_check.reshape(-1))
keep = y_check.reshape(-1) != IGNORE_INDEX
by_hand = F.cross_entropy(logits.view(-1, vocab_size)[keep], y_check.reshape(-1)[keep])
assert torch.allclose(as_model_computes_it, by_hand), (
    f"cross_entropy is not ignoring the mask: {as_model_computes_it:.6f} vs {by_hand:.6f}"
)

print(f"x[0] {x_check[0].tolist()}")
print(f"y[0] {y_check[0].tolist()}   <- -100 wherever there is nothing to predict")
print(f"\nmasked loss {as_model_computes_it:.4f}, identical when computed by hand over "
      f"the {int(keep.sum())} unmasked positions")

### Splitting on whole windows

The split unit is the **window**, which makes leakage structurally impossible: an exchange
lives in exactly one window, so it cannot appear in both halves.

The split is random rather than contiguous, for the reason 6_1 gives — these are discrete
samples with no overlap, unlike notebook 4's sliding windows, so a random split is safe and
far more representative than taking the tail of one chat.

Note how small the validation set is: 5% of 1,191 windows is about 60. Packing bought a 3.4x
speedup and paid for it here, in the resolution of the validation signal.

In [ ]:
inputs, targets = pad_and_mask(windows, block_size, padding_token)

generator = torch.Generator().manual_seed(3647)
order = torch.randperm(len(windows), generator=generator)
split_index = int((1 - val_fraction) * len(windows))
train_index, val_index = order[:split_index], order[split_index:]

train_loader = DataLoader(
    TensorDataset(inputs[train_index].to(device), targets[train_index].to(device)),
    batch_size=batch_size, shuffle=True,
)
val_loader = DataLoader(
    TensorDataset(inputs[val_index].to(device), targets[val_index].to(device)),
    batch_size=batch_size, shuffle=False,
)

scored = int((targets != IGNORE_INDEX).sum())
print(f"{len(windows):,} windows -> {len(train_index):,} train / {len(val_index):,} validation")
print(f"{len(train_loader)} training batches per epoch, {len(val_loader)} validation batches")
print(f"{scored:,} of {targets.numel():,} target positions carry loss "
      f"({scored / targets.numel():.1%})")

## The model

Built from the config read earlier, so it cannot disagree with the weights going into it.

Two things the original version of this cell got wrong, both fatal before a step ran: it
passed `ignore_index=` to `GPTLanguageModel`, which is not a constructor argument, and it
called `torch.compile(model)` *before* `load_state_dict`. Compilation prefixes every
`state_dict` key with `_orig_mod.`, and the pre-trained checkpoint has no such prefix, so the
load fails on every key at once. Following notebook 4, compilation is not used here.

In [ ]:
model = GPTLanguageModel(**config).to(device)
model.load_state_dict(checkpoint["model_state_dict"])   # raises on any key or shape mismatch
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

random_loss = math.log(vocab_size)
model.eval()
with torch.no_grad():
    x_probe, y_probe = next(iter(val_loader))
    _, loaded_loss = model(x_probe, y_probe)
model.train()

print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.2f} M parameters, "
      f"restored from {PRETRAINED_CHECKPOINT.name}")
print(f"loss on a validation batch : {loaded_loss:.4f}")
print(f"an untrained model would be: {random_loss:.4f}   (ln of vocab_size {vocab_size})")

assert loaded_loss < random_loss - 0.5, (
    f"loss {loaded_loss:.4f} is too close to random ({random_loss:.4f}) - the checkpoint "
    "weights did not load, or loaded into the wrong model"
)
print("\nthe checkpoint carries real pre-trained weights")

## Fine-tuning

`estimate_loss` averages a bounded number of batches and restores whatever mode the model was
in. `save_checkpoint` creates its parent directory and unwraps `_orig_mod.` defensively, so
the checkpoint stays loadable whether or not compilation was used.

It writes to `output/fine_tuning/with_context/` rather than `run_1/`, which is 6_1's. The
original wrote both notebooks to the same `run_1/` directory, so whichever ran second
silently overwrote the other's work.

In [ ]:
@torch.no_grad()
def estimate_loss(model, loaders: dict, eval_batches: int) -> dict:
    """Average loss over a fixed number of batches from each loader."""
    was_training = model.training
    model.eval()
    try:
        results = {}
        for split, loader in loaders.items():
            losses = []
            for _, (x, y) in zip(range(eval_batches), loader):
                _, loss = model(x, y)
                losses.append(loss.item())
            results[split] = sum(losses) / len(losses)
        return results
    finally:
        model.train(was_training)


def save_checkpoint(model, optimizer, epoch, step, loss, file_path) -> None:
    """Write a checkpoint that a plain GPTLanguageModel can load."""
    file_path = Path(file_path)
    file_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            "epoch": epoch,
            "step": step,
            "loss": loss,
            "model_state_dict": getattr(model, "_orig_mod", model).state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "config": config,
        },
        file_path,
    )


def synchronize(device: str) -> None:
    if device == "cuda":
        torch.cuda.synchronize()
    elif device == "mps":
        torch.mps.synchronize()

In [ ]:
loaders = {"train": train_loader, "val": val_loader}
history = {"step": [], "train": [], "val": []}

steps_per_epoch = len(train_loader)
total_steps = max_steps if max_steps is not None else steps_per_epoch * max_epochs
global_step, best_val, best_step, stale_evals, stop = 0, float("inf"), -1, 0, False

model.train()
start_time = time.perf_counter()

for epoch in range(max_epochs):
    if stop:
        break
    for x_batch, y_batch in train_loader:
        if max_steps is not None and global_step >= max_steps:
            stop = True
            break

        if global_step % eval_interval == 0:
            losses = estimate_loss(model, loaders, eval_batches)
            history["step"].append(global_step)
            history["train"].append(losses["train"])
            history["val"].append(losses["val"])

            if losses["val"] < best_val:
                best_val, best_step, stale_evals = losses["val"], global_step, 0
                save_checkpoint(model, optimizer, epoch, global_step, best_val,
                                OUTPUT_CHECKPOINT)
                marker = "  <- saved"
            else:
                stale_evals += 1
                marker = ""

            print(f"epoch {epoch:>2}  step {global_step:>4}/{total_steps}  "
                  f"train {losses['train']:.4f}  val {losses['val']:.4f}{marker}")

            if stale_evals >= early_stop_patience:
                print(f"\nno validation improvement for {stale_evals} evaluations - stopping")
                stop = True
                break

        _, loss = model(x_batch, y_batch)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()
        global_step += 1

synchronize(device)
print(f"\n{global_step} steps in {(time.perf_counter() - start_time) / 60:.1f} min")
print(f"best validation loss {best_val:.4f} at step {best_step}")
print(f"saved to {OUTPUT_CHECKPOINT.resolve()}")

## The loss curve

In [ ]:
try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError:
    print("matplotlib is not installed - `pip install matplotlib` for the chart.\n")
    print(f"{'step':>8}{'train':>10}{'val':>10}")
    for s, t, v in zip(history["step"], history["train"], history["val"]):
        print(f"{s:>8}{t:>10.4f}{v:>10.4f}")
else:
    figure, axis = plt.subplots(figsize=(9, 5))
    axis.plot(history["step"], history["train"], marker="o", label="train")
    axis.plot(history["step"], history["val"], marker="o", label="validation")
    if best_step >= 0:
        axis.axvline(best_step, color="grey", linestyle="--", linewidth=1,
                     label=f"best val @ {best_step}")
    axis.set_xlabel("step")
    axis.set_ylabel("cross-entropy loss")
    axis.set_title("Fine-tuning with context")
    axis.legend()
    axis.grid(alpha=0.3)
    plt.show()

## Does the context actually help?

This is the question the notebook exists to answer, and it is where the obvious measurement
gives the wrong answer.

**The tempting version.** Group every scored position by which exchange it belongs to, and
compare: the fifth exchange in a window has four exchanges of history in front of it, the
first has none. Run that and the loss falls steeply from exchange 1 to exchange 6 — a
beautiful, monotonic, completely misleading curve.

**Why it is misleading.** A window is 256 tokens. To fit six exchanges into it, all six have
to be *short* — and short exchanges are far easier regardless of context (measured on this
corpus: replies of 6-10 tokens run at perplexity 3.6, replies over 40 at 14.3). So position 6
is populated almost entirely by easy exchanges. The curve is mostly measuring length, and
attributing it to context.

**The controlled version.** Take the *same* exchange and score it twice: once sitting in its
window with history in front of it, once alone with none. Same tokens, same length, same
content — the only thing that changes is whether the history is there. That difference is the
context effect, and nothing else.

Both are computed below, because seeing the gap between them is the point. This is the same
trap notebook 3 documents for fused attention, where an uncontrolled benchmark claimed 23.5x
for something that was really 1.1x.

In [ ]:
@torch.no_grad()
def token_losses(ids: list[int]) -> list[float]:
    """Per-token loss for one sequence, with no averaging."""
    x = torch.tensor([ids[:-1]], device=device)
    y = torch.tensor([ids[1:]], device=device)
    logits, _ = model(x)
    return F.cross_entropy(
        logits.view(-1, vocab_size), y.view(-1), reduction="none"
    ).tolist()


was_training = model.training
model.eval()

by_position = {k: [0.0, 0] for k in range(6)}     # the tempting, confounded measurement
with_context = [0.0, 0]                            # the controlled one
without_context = [0.0, 0]
helped = total = 0

for row in val_index.tolist():
    ids = windows[row]
    full = token_losses(ids)
    opens = [i for i, t in enumerate(ids) if t == start_token][::2]

    for position, start in enumerate(opens):
        end = opens[position + 1] if position + 1 < len(opens) else len(ids)

        if position < 6:                           # naive: bucket by slot in the window
            span = full[max(0, start - 1):end - 1]
            if span:
                by_position[position][0] += sum(span)
                by_position[position][1] += len(span)

        if position == 0 or end - start < 8:
            continue                               # nothing to compare the first one against

        # Controlled: the identical exchange, with and without the history in front.
        inside = full[start - 1:end - 1]
        standalone = token_losses(ids[start:end])
        n = min(len(inside), len(standalone))
        with_context[0] += sum(inside[:n]);      with_context[1] += n
        without_context[0] += sum(standalone[:n]); without_context[1] += n
        helped += sum(inside[:n]) < sum(standalone[:n])
        total += 1

model.train(was_training)

print("the tempting measurement - mean loss by position in the window:\n")
print(f"{'exchange':>9} {'loss':>7} {'ppl':>7}   history in front")
naive = {k: t / c for k, (t, c) in by_position.items() if c}
for position, value in naive.items():
    print(f"{position + 1:>9} {value:>7.4f} {math.exp(value):>7.1f}   "
          f"{position} exchange(s)  " + "#" * int((value - min(naive.values())) * 40 + 1))
print(f"\n  looks like a {naive[0] - naive[max(naive)]:.2f} nat improvement - but windows "
      "holding 6 exchanges\n  can only hold *short* ones, and short exchanges are easier "
      "whatever precedes them.")

inside_mean = with_context[0] / with_context[1]
alone_mean = without_context[0] / without_context[1]
print(f"\n\nthe controlled measurement - the SAME {total} exchanges, scored twice:\n")
print(f"  with the window history in front : {inside_mean:.4f}  (ppl {math.exp(inside_mean):5.1f})")
print(f"  alone, no history                : {alone_mean:.4f}  (ppl {math.exp(alone_mean):5.1f})")
print(f"  context is worth                 : {alone_mean - inside_mean:+.4f} nats "
      f"({1 - math.exp(inside_mean) / math.exp(alone_mean):.0%} of the perplexity)")
print(f"  it helped on {helped}/{total} exchanges ({helped / total:.0%})")

## Talking to it, with history

This is what the packed training data buys. `Conversation` keeps every turn so far and feeds
the whole transcript back as the prompt, exactly matching the shape of a training window:

```
<|startoftext|>You<|separator|>first message<|endoftext|>
<|startoftext|>Assistant<|separator|>first reply<|endoftext|>
<|startoftext|>You<|separator|>second message<|endoftext|>
<|startoftext|>Assistant<|separator|>            <- the model continues from here
```

Two details that are easy to get wrong:

**Truncate from the left.** When the transcript outgrows `block_size`, the *oldest* turns are
dropped, never the newest — the current question has to survive. `model.generate` also crops
internally, but doing it here keeps the prompt aligned to a turn boundary rather than slicing
mid-sentence.

**Use `model.generate`.** The original hand-rolled a `while True` loop calling
`generate(max_new_tokens=1)` one token at a time, re-running the whole forward pass per token.
`generate` takes `eos_token_id`, stops on it, and is bounded by `max_new_tokens` either way.

In [ ]:
class Conversation:
    """A multi-turn chat that feeds its own history back as context."""

    def __init__(self, model, tokenizer, block_size: int, device: str) -> None:
        self.model, self.tokenizer = model, tokenizer
        self.block_size, self.device = block_size, device
        self.turns: list[tuple[str, str]] = []

    def _prompt_ids(self) -> list[int]:
        transcript = "".join(
            f"{START_OF_TEXT}{role}{SEPARATOR}{text}{END_OF_TEXT}" for role, text in self.turns
        ) + f"{START_OF_TEXT}{ASSISTANT_ROLE}{SEPARATOR}"
        ids = self.tokenizer.encode(transcript, allowed_special="all")

        # Drop whole turns from the front until the prompt fits, so the newest question
        # always survives and the prompt still starts on a turn boundary.
        while len(ids) > self.block_size - 8 and len(self.turns) > 1:
            self.turns.pop(0)
            return self._prompt_ids()
        return ids

    def say(self, message: str, max_new_tokens: int = 40,
            temperature: float = 0.8) -> tuple[str, bool]:
        self.turns.append((USER_ROLE, message))
        prompt_ids = self._prompt_ids()

        output = self.model.generate(
            input_tokens=torch.tensor([prompt_ids], dtype=torch.long, device=self.device),
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            eos_token_id=eos_token,
        )
        generated = output[0, len(prompt_ids):].tolist()
        stopped = eos_token in generated
        if stopped:
            generated = generated[:generated.index(eos_token)]

        reply = self.tokenizer.decode(generated)
        self.turns.append((ASSISTANT_ROLE, reply))
        return reply, stopped


torch.manual_seed(11)
chat = Conversation(model, tokenizer, block_size, device)

for message in ("hey", "are you coming tomorrow", "what time", "ok see you"):
    reply, stopped = chat.say(message)
    print(f"You       : {message}")
    print(f"Assistant : {reply}" + ("" if stopped else "   <- hit the token cap"))
print(f"\ntranscript now {len(chat.turns)} turns, "
      f"{len(chat._prompt_ids())} tokens of a {block_size}-token window")

### Where to go next

- **Record chat boundaries in notebook 5.** The 1.26% of windows that straddle two
  conversations are the one structural defect here, and packing cannot fix it without knowing
  where each chat ends. A sidecar file of per-chat sample counts would remove it entirely.
- **Mask the prompts.** Setting every `You` turn to `IGNORE_INDEX` concentrates the gradient
  on replies. With packing there are several prompts per window, so this matters more here
  than it did in 6_1.
- **Never report the by-position curve on its own.** It is the number this notebook would
  most like to be true, and it is inflated several-fold by exchange length. Any claim about
  context needs the controlled comparison behind it — same exchange, with history and
  without.
- **Most of what packing bought here was throughput, not context.** The 3.4x fewer steps is
  large and certain; the context benefit is real but small at this scale. Both are worth
  having, and they should not be quoted as one number.
- **The ceiling is still the data.** 13.8M parameters over ~176k distinct tokens is roughly
  three orders of magnitude short of compute-optimal. Context changes what the model *can*
  condition on, not how much it knows.